# CNN-BiLSTM Activity Classifier

A CNN + bidirectional LSTM with attention pooling, trained and evaluated with
the same 5-fold subject-wise protocol as `cnn_model.ipynb`. See cell 3 for the
rationale versus the plain CNN.

In [1]:
# ============================================================================
#  CNN-BiLSTM activity classifier - 5-fold subject-wise cross-validation
# ----------------------------------------------------------------------------
#  Data:  balanced_folds/fold_<i>/  built by data_processing.ipynb
#         X_* (N, 128, 6) float32 - 4 s segments, 32 Hz, acc x/y/z + gyro x/y/z
#         y_* (N,) int8            - class 0-6
#  Runs:  five independent trainings. Each model sees only its own fold's
#         train users, early-stops on its own val users, and predicts its own
#         test users. The five test sets are disjoint and cover all 56 users.
#
#  Why this model over the plain CNN in cnn_model.ipynb: that model's global
#  average pool collapses the 128-step sequence into a single order-blind
#  summary right after 3 conv layers, so it cannot use *how* a window evolves
#  over time - only what magnitudes appear in it. Walking/Running/Bicycling are
#  mostly distinguished by cadence (a periodic pattern in time), and its
#  confusion matrix showed exactly that failure (cnn_results/pooled.json).
#  Here the same conv front end feeds a BiLSTM, which encodes each timestep in
#  the context of the whole window before an attention layer pools it - order
#  is preserved all the way to the classifier head.
#
#  NOTE on torch/numpy: torch 2.2.0 here is compiled against NumPy 1.x and the
#  installed NumPy is 2.4.6, so torch.from_numpy() and tensor.numpy() both raise
#  "Numpy is not available". Everything below therefore crosses the boundary
#  through the buffer protocol (torch.frombuffer / .tolist()), which does not
#  touch the numpy C-API. Upgrading torch would also fix it, but the CUDA 12.1
#  build matches this driver (525.89.02) and a newer build may not.
# ============================================================================
import json
import os
import time

import numpy as np
import torch
import torch.nn as nn

DATA_DIR    = "balanced_folds"
RESULTS_DIR = "cnn_bilstm_results"
N_FOLDS     = 5
N_CLASSES   = 7
SEG_LEN     = 128
N_CHANNELS  = 6

BATCH       = 512
EVAL_BATCH  = 4096
MAX_EPOCHS  = 20
LR          = 1e-3
WEIGHT_DECAY= 1e-4
GRAD_CLIP   = 5.0         # LSTMs are more prone to exploding gradients than pure conv nets
PATIENCE     = 4          # epochs without val macro-F1 improvement
SEED        = 42

CLASS_NAMES = ["Lying down", "Sitting", "Walking", "Running",
               "Bicycling", "Standing in place", "Standing and moving"]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {DEVICE}"
      + (f" ({torch.cuda.get_device_name(0)})" if DEVICE.type == "cuda" else ""))
print(f"torch {torch.__version__} | numpy {np.__version__}")

_TORCH_DTYPE = {"float32": torch.float32, "float64": torch.float64,
                "int8": torch.int8, "int16": torch.int16,
                "int32": torch.int32, "int64": torch.int64, "bool": torch.bool}


def to_torch(a, device=None):
    """numpy array -> torch tensor without using the (broken) numpy bridge."""
    a = np.ascontiguousarray(a)
    t = torch.frombuffer(memoryview(a.reshape(-1)),
                         dtype=_TORCH_DTYPE[a.dtype.name]).reshape(a.shape)
    return t.to(device) if device is not None else t.clone()


def set_seed(s):
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/srijib/miniconda3/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/home/srijib/miniconda3/lib/python3.11/site-packages/traitlets/config/application.py", line 1080, in launch_instance
    app.start()
  File "/home/srijib/miniconda3/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 758, in start
    self.io_loop.start()
  File "/home/sr

device: cuda (NVIDIA RTX A4500)
torch 2.2.0 | numpy 2.4.6


In [2]:
# ---------------------------------------------------------------------------
#  Metrics, computed from a confusion matrix
# ---------------------------------------------------------------------------
#  Everything we need - accuracy, macro-F1, balanced accuracy, Cohen's kappa -
#  is a function of the 7x7 confusion matrix, so predictions never have to be
#  converted to numpy (which the torch/numpy mismatch would block anyway).
#  Only 49 integers cross back from the GPU per evaluation.
# ---------------------------------------------------------------------------

def confusion(y_true, y_pred, n=N_CLASSES):
    """7x7 confusion matrix on-device. Rows = true class, cols = predicted."""
    k = (y_true.long() * n + y_pred.long())
    return torch.bincount(k, minlength=n * n).reshape(n, n)


def metrics_from_cm(cm):
    """accuracy, macro-F1, balanced accuracy, Cohen's kappa + per-class detail."""
    cm = np.asarray(cm.tolist(), dtype=np.float64)      # 49 numbers, no bridge
    total = cm.sum()
    if total == 0:
        return {}
    tp = np.diag(cm)
    row = cm.sum(1)                                      # support per true class
    col = cm.sum(0)                                      # predictions per class

    accuracy = tp.sum() / total

    with np.errstate(divide="ignore", invalid="ignore"):
        recall = np.where(row > 0, tp / np.maximum(row, 1), 0.0)
        precision = np.where(col > 0, tp / np.maximum(col, 1), 0.0)
        denom = precision + recall
        f1 = np.where(denom > 0, 2 * precision * recall / np.maximum(denom, 1e-12), 0.0)

    # macro averages ignore classes absent from the ground truth of this split
    present = row > 0
    macro_f1 = f1[present].mean()
    balanced_accuracy = recall[present].mean()

    p_e = float((row * col).sum()) / (total * total)     # chance agreement
    kappa = (accuracy - p_e) / (1 - p_e) if p_e < 1 else 0.0

    return {"accuracy": float(accuracy), "macro_f1": float(macro_f1),
            "balanced_accuracy": float(balanced_accuracy), "kappa": float(kappa),
            "per_class": {"precision": precision, "recall": recall, "f1": f1,
                          "support": row.astype(np.int64)},
            "cm": cm.astype(np.int64)}


def print_per_class(m, title="per-class"):
    pc = m["per_class"]
    print(f"\n  {title}")
    print(f"    {'idx':>3}  {'class':22s} {'precision':>10s} {'recall':>8s} "
          f"{'f1':>8s} {'support':>10s}")
    for k in range(N_CLASSES):
        print(f"    {k:>3}  {CLASS_NAMES[k]:22s} {pc['precision'][k]:10.3f} "
              f"{pc['recall'][k]:8.3f} {pc['f1'][k]:8.3f} {pc['support'][k]:10,}")

In [3]:
# ---------------------------------------------------------------------------
#  Loading a fold onto the GPU
# ---------------------------------------------------------------------------
#  Each fold is ~2.6 GB across train/val/test, which fits comfortably on a 20 GB
#  card, so the whole fold is resident and batching is plain tensor indexing -
#  no DataLoader, no per-batch host->device copies.
#
#  Normalisation uses THIS fold's norm_mean / norm_std, which were fitted on its
#  real (non-augmented) training segments only. Using another fold's statistics,
#  or statistics computed over all data, would leak test users into training.
# ---------------------------------------------------------------------------

def load_fold(i, device=DEVICE):
    d = f"{DATA_DIR}/fold_{i}"
    mean = to_torch(np.load(f"{d}/norm_mean.npy"), device)      # (6,)
    std = to_torch(np.load(f"{d}/norm_std.npy"), device)

    out = {}
    for part in ("train", "val", "test"):
        X = np.load(f"{d}/X_{part}.npy")                        # (N, 128, 6)
        y = np.load(f"{d}/y_{part}.npy")
        Xt = to_torch(X, device)
        del X
        Xt = (Xt - mean) / std                                  # broadcast over channels
        Xt = Xt.permute(0, 2, 1).contiguous()                   # -> (N, 6, 128) for Conv1d
        out[part] = (Xt, to_torch(y, device).long())
        del y

    with open(f"{d}/class_weights.json") as fh:
        out["class_weights"] = to_torch(
            np.asarray(json.load(fh)["class_weights"], dtype="float32"), device)
    out["users"] = json.load(open(f"{d}/report.json"))["users"]
    return out


def describe_fold(f, i):
    tr, va, te = f["train"][0], f["val"][0], f["test"][0]
    used = sum(t.numel() * t.element_size() for t in (tr, va, te)) / 1e9
    print(f"  fold {i}: train {tuple(tr.shape)}  val {tuple(va.shape)}  "
          f"test {tuple(te.shape)}  [{used:.2f} GB on {DEVICE}]")
    print(f"          users  train {len(f['users']['train'])}  "
          f"val {len(f['users']['val'])}  test {len(f['users']['test'])}")

In [4]:
# ---------------------------------------------------------------------------
#  The model: a two-scale 1D CNN feeding a BiLSTM with attention pooling
# ---------------------------------------------------------------------------
#  Front end - unchanged from cnn_model.ipynb's HARCNN: two parallel first-layer
#  branches read the window at different time scales, then their features are
#  concatenated and reduced:
#     k=5  (~0.16 s) - fast transients: a footfall, a jolt
#     k=21 (~0.66 s) - a whole gait or pedal cycle
#  Two pooling stages bring 128 timesteps down to 32, trading temporal
#  resolution for a sequence length the LSTM can iterate over cheaply.
#
#  Back end - a bidirectional LSTM over the 32-step conv feature sequence. Each
#  output position is a summary of the *entire* window (both directions), so by
#  the time attention pools it, order and periodicity are already encoded -
#  unlike HARCNN's global-average-pool, which is order-blind by construction.
#
#  Attention pooling instead of mean/last-hidden: a fixed phase offset (the
#  segments are not gait-aligned) means the informative part of the window
#  moves around, so a learned per-position weight beats either a flat average
#  or committing to one timestep.
# ---------------------------------------------------------------------------

class HARCNNBiLSTM(nn.Module):
    def __init__(self, in_ch=N_CHANNELS, n_classes=N_CLASSES,
                 lstm_hidden=128, lstm_layers=1, p_drop=0.4):
        super().__init__()
        self.fast = nn.Sequential(
            nn.Conv1d(in_ch, 48, kernel_size=5, padding=2),
            nn.BatchNorm1d(48), nn.ReLU(inplace=True))
        self.slow = nn.Sequential(
            nn.Conv1d(in_ch, 48, kernel_size=21, padding=10),
            nn.BatchNorm1d(48), nn.ReLU(inplace=True))
        self.conv_body = nn.Sequential(
            nn.MaxPool1d(2),                                   # 128 -> 64
            nn.Conv1d(96, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128), nn.ReLU(inplace=True),
            nn.MaxPool1d(2),                                   # 64 -> 32
            nn.Conv1d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128), nn.ReLU(inplace=True))        # -> (B, 128, 32)

        self.lstm = nn.LSTM(input_size=128, hidden_size=lstm_hidden,
                             num_layers=lstm_layers, batch_first=True,
                             bidirectional=True,
                             dropout=p_drop if lstm_layers > 1 else 0.0)
        lstm_out = lstm_hidden * 2                             # bidirectional

        self.attn = nn.Linear(lstm_out, 1)                     # additive attention pooling
        self.head = nn.Sequential(
            nn.Dropout(p_drop),
            nn.Linear(lstm_out, 128), nn.ReLU(inplace=True),
            nn.Dropout(p_drop),
            nn.Linear(128, n_classes))

    def forward(self, x):                                      # x: (B, 6, 128)
        x = torch.cat([self.fast(x), self.slow(x)], dim=1)     # (B, 96, 128)
        x = self.conv_body(x)                                  # (B, 128, 32)
        x = x.permute(0, 2, 1)                                 # (B, 32, 128) for LSTM
        seq, _ = self.lstm(x)                                  # (B, 32, 2*hidden)
        weights = torch.softmax(self.attn(seq), dim=1)         # (B, 32, 1)
        pooled = (seq * weights).sum(dim=1)                    # (B, 2*hidden)
        return self.head(pooled)


m = HARCNNBiLSTM()
print(f"HARCNNBiLSTM: {sum(p.numel() for p in m.parameters()):,} parameters")
print(f"  forward check: {tuple(m(torch.zeros(2, N_CHANNELS, SEG_LEN)).shape)} "
      f"(expect (2, {N_CLASSES}))")
del m

HARCNNBiLSTM: 417,384 parameters
  forward check: (2, 7) (expect (2, 7))


In [5]:
# ---------------------------------------------------------------------------
#  Train / evaluate one fold
# ---------------------------------------------------------------------------
#  Early stopping watches validation MACRO-F1, not loss or accuracy. Validation
#  is deliberately left at the natural class distribution (~44% Sitting, 0.4%
#  Running), so loss and accuracy are both dominated by the majority classes and
#  would happily select a model that never predicts Running at all.
#
#  Gradient clipping is the one addition over cnn_model.ipynb's loop: an LSTM
#  backpropagating through 32 timesteps is more prone to occasional gradient
#  spikes than a pure feed-forward conv net.
# ---------------------------------------------------------------------------

@torch.no_grad()
def evaluate(model, X, y, batch=EVAL_BATCH, return_preds=False):
    model.eval()
    cm = torch.zeros(N_CLASSES, N_CLASSES, dtype=torch.long, device=X.device)
    preds = [] if return_preds else None
    for s in range(0, len(X), batch):
        p = model(X[s:s + batch]).argmax(1)
        cm += confusion(y[s:s + batch], p)
        if return_preds:
            preds.append(p)
    m = metrics_from_cm(cm)
    return (m, torch.cat(preds)) if return_preds else (m, None)


def train_fold(i, verbose=True):
    set_seed(SEED + i)
    f = load_fold(i)
    if verbose:
        describe_fold(f, i)
    Xtr, ytr = f["train"]
    Xva, yva = f["val"]
    Xte, yte = f["test"]

    model = HARCNNBiLSTM().to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max",
                                                       factor=0.5, patience=1)
    lossf = nn.CrossEntropyLoss(weight=f["class_weights"])

    best = {"macro_f1": -1.0}
    best_state, best_epoch, since = None, -1, 0
    n = len(Xtr)
    history = []

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        t0 = time.time()
        perm = torch.randperm(n, device=DEVICE)
        run_loss = 0.0
        for s in range(0, n, BATCH):
            idx = perm[s:s + BATCH]
            opt.zero_grad(set_to_none=True)
            loss = lossf(model(Xtr[idx]), ytr[idx])
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()
            run_loss += loss.item() * len(idx)
        vm, _ = evaluate(model, Xva, yva)
        sched.step(vm["macro_f1"])
        history.append({"epoch": epoch, "loss": run_loss / n, **{k: vm[k] for k in
                        ("accuracy", "macro_f1", "balanced_accuracy", "kappa")}})
        if verbose:
            print(f"    epoch {epoch:2d}  loss {run_loss / n:.4f}  "
                  f"val macro-F1 {vm['macro_f1']:.4f}  acc {vm['accuracy']:.4f}  "
                  f"bal-acc {vm['balanced_accuracy']:.4f}  ({time.time() - t0:.0f}s)")

        if vm["macro_f1"] > best["macro_f1"] + 1e-5:
            best, best_epoch, since = vm, epoch, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            since += 1
            if since >= PATIENCE:
                if verbose:
                    print(f"    early stop (no val improvement for {PATIENCE} epochs)")
                break

    model.load_state_dict(best_state)                 # restore the best epoch
    test_m, test_p = evaluate(model, Xte, yte, return_preds=True)
    if verbose:
        print(f"    best epoch {best_epoch} (val macro-F1 {best['macro_f1']:.4f})"
              f"  ->  TEST macro-F1 {test_m['macro_f1']:.4f}  "
              f"acc {test_m['accuracy']:.4f}  bal-acc {test_m['balanced_accuracy']:.4f}  "
              f"kappa {test_m['kappa']:.4f}")

    out = {"fold": i, "best_epoch": best_epoch, "history": history,
           "val": {k: best[k] for k in ("accuracy", "macro_f1",
                                        "balanced_accuracy", "kappa")},
           "test": test_m,
           "y_true": yte.cpu().tolist(), "y_pred": test_p.cpu().tolist()}
    del f, Xtr, ytr, Xva, yva, Xte, yte, model, best_state
    torch.cuda.empty_cache()
    return out

In [6]:
# ---------------------------------------------------------------------------
#  Run all five folds
# ---------------------------------------------------------------------------
results = []
t_start = time.time()
for i in range(N_FOLDS):
    print(f"\n{'=' * 72}\nFOLD {i}\n{'=' * 72}")
    results.append(train_fold(i))
print(f"\nall folds complete in {(time.time() - t_start) / 60:.1f} min")

os.makedirs(RESULTS_DIR, exist_ok=True)
json.dump([{k: v for k, v in r.items() if k not in ("test", "y_true", "y_pred")}
           | {"test": {k: v for k, v in r["test"].items()
                       if k not in ("per_class", "cm")}}
           for r in results], open(f"{RESULTS_DIR}/folds.json", "w"), indent=1)
json.dump({str(r["fold"]): {"y_true": r["y_true"], "y_pred": r["y_pred"]}
           for r in results}, open(f"{RESULTS_DIR}/predictions.json", "w"))
print(f"saved -> {RESULTS_DIR}/folds.json, {RESULTS_DIR}/predictions.json")


FOLD 0


  fold 0: train (333576, 6, 128)  val (260994, 6, 128)  test (241576, 6, 128)  [2.57 GB on cuda]
          users  train 36  val 8  test 12


    epoch  1  loss 1.4548  val macro-F1 0.2515  acc 0.4535  bal-acc 0.3426  (9s)


    epoch  2  loss 1.3550  val macro-F1 0.2584  acc 0.4380  bal-acc 0.3422  (8s)


    epoch  3  loss 1.3161  val macro-F1 0.2475  acc 0.4257  bal-acc 0.3427  (8s)


...

    epoch  4  loss 1.3047  val macro-F1 0.2158  acc 0.3789  bal-acc 0.3096  (8s)


    epoch  5  loss 1.2579  val macro-F1 0.2014  acc 0.3189  bal-acc 0.2873  (8s)


    epoch  6  loss 1.2389  val macro-F1 0.1943  acc 0.3179  bal-acc 0.2902  (8s)
    early stop (no val improvement for 4 epochs)


    best epoch 2 (val macro-F1 0.2212)  ->  TEST macro-F1 0.2870  acc 0.3525  bal-acc 0.4692  kappa 0.1684

all folds complete in 5.6 min


saved -> cnn_bilstm_results/folds.json, cnn_bilstm_results/predictions.json


In [7]:
# ---------------------------------------------------------------------------
#  Per-fold results, and mean +/- standard deviation across the five folds
# ---------------------------------------------------------------------------
METRICS = [("accuracy", "Accuracy"), ("macro_f1", "Macro F1"),
           ("balanced_accuracy", "Balanced accuracy"), ("kappa", "Cohen's kappa")]

vals = {k: np.array([r["test"][k] for r in results]) for k, _ in METRICS}

print("TEST metrics per fold\n")
print(f"  {'fold':>4} {'n_test':>10} {'epochs':>7}  "
      + "".join(f"{lab:>19s}" for _, lab in METRICS))
for r in results:
    print(f"  {r['fold']:>4} {int(r['test']['per_class']['support'].sum()):>10,} "
          f"{r['best_epoch']:>7}  "
          + "".join(f"{r['test'][k]:19.4f}" for k, _ in METRICS))

print(f"\n  {'mean':>4} {'':>10} {'':>7}  "
      + "".join(f"{vals[k].mean():19.4f}" for k, _ in METRICS))
print(f"  {'std':>4} {'':>10} {'':>7}  "
      + "".join(f"{vals[k].std(ddof=1):19.4f}" for k, _ in METRICS))

print("\n" + "-" * 72)
print("SUMMARY  (mean +/- std over 5 subject-wise folds)\n")
for k, lab in METRICS:
    print(f"  {lab:20s} {vals[k].mean():.4f}  +/-  {vals[k].std(ddof=1):.4f}"
          f"     [min {vals[k].min():.4f}, max {vals[k].max():.4f}]")

# for reference - the plain two-scale CNN (cnn_model.ipynb, no LSTM stage) pooled to:
#   accuracy 0.3664   macro-F1 0.2594   balanced-accuracy 0.3417   kappa 0.1592
print("\n  for reference, on this data:")
print("    always-predict-Sitting        accuracy 0.441   macro-F1 0.087   kappa 0.000")
print("    only the 2 majority classes   accuracy 0.787   macro-F1 0.258")
print("    HARCNN (cnn_model.ipynb)      accuracy 0.366   macro-F1 0.259   kappa 0.159")

TEST metrics per fold

  fold     n_test  epochs             Accuracy           Macro F1  Balanced accuracy      Cohen's kappa
     0    241,576       2               0.4057             0.2586             0.4121             0.1989
     1    280,351       1               0.3596             0.2259             0.3902             0.1561
     2    371,308       6               0.4384             0.3229             0.3759             0.2261
     3    306,925       8               0.3786             0.2395             0.2942             0.1228
     4    133,255       2               0.3525             0.2870             0.4692             0.1684

  mean                                  0.3870             0.2668             0.3883             0.1745
   std                                  0.0354             0.0389             0.0635             0.0397

------------------------------------------------------------------------
SUMMARY  (mean +/- std over 5 subject-wise folds)

  Accuracy         

In [8]:
# ---------------------------------------------------------------------------
#  Pooled result across all five folds
# ---------------------------------------------------------------------------
#  Every user is tested exactly once, so concatenating the five prediction sets
#  gives ONE estimate over all 56 users. This is preferred to averaging the five
#  fold scores, because the test folds differ in size by nearly 3x (133k to
#  371k segments) and equal weighting would misrepresent that.
# ---------------------------------------------------------------------------
yt = torch.tensor([v for r in results for v in r["y_true"]])
yp = torch.tensor([v for r in results for v in r["y_pred"]])
pooled = metrics_from_cm(confusion(yt, yp))

print(f"pooled over {len(yt):,} test segments from all 56 users\n")
for k, lab in METRICS:
    print(f"  {lab:20s} {pooled[k]:.4f}")
print_per_class(pooled, "per-class (pooled)")

print("\n  confusion matrix, row-normalised (rows = true class)\n")
cm = pooled["cm"].astype(np.float64)
cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
print("    " + " " * 24 + "".join(f"{i:>8d}" for i in range(N_CLASSES)))
for i in range(N_CLASSES):
    print(f"    {i} {CLASS_NAMES[i]:22s}"
          + "".join(f"{cmn[i, j]:8.3f}" for j in range(N_CLASSES)))

json.dump({k: pooled[k] for k, _ in METRICS}
          | {"per_class_f1": pooled["per_class"]["f1"].tolist(),
             "per_class_recall": pooled["per_class"]["recall"].tolist(),
             "per_class_support": pooled["per_class"]["support"].tolist(),
             "confusion_matrix": pooled["cm"].tolist(),
             "class_names": CLASS_NAMES},
          open(f"{RESULTS_DIR}/pooled.json", "w"), indent=1)
print(f"\nsaved -> {RESULTS_DIR}/pooled.json")

pooled over 1,333,415 test segments from all 56 users

  Accuracy             0.3936
  Macro F1             0.2647
  Balanced accuracy    0.3523
  Cohen's kappa        0.1746

  per-class (pooled)
    idx  class                   precision   recall       f1    support
      0  Lying down                  0.465    0.748    0.573    461,429
      1  Sitting                     0.575    0.172    0.265    588,134
      2  Walking                     0.413    0.444    0.428     94,632
      3  Running                     0.021    0.187    0.038      4,985
      4  Bicycling                   0.206    0.586    0.305     20,827
      5  Standing in place           0.054    0.204    0.085     35,709
      6  Standing and moving         0.215    0.126    0.159    127,699

  confusion matrix, row-normalised (rows = true class)

                                   0       1       2       3       4       5       6
    0 Lying down               0.748   0.104   0.014   0.021   0.020   0.058   0.034


In [9]:
# ---------------------------------------------------------------------------
#  TABLE 1 - per-class metrics, pooled over all five folds
# ---------------------------------------------------------------------------
#  Every user is tested exactly once across the five folds, so concatenating the
#  predictions gives one estimate over all 56 users.
#
#  "accuracy", "macro F1" and "balanced accuracy" are aggregate metrics over all
#  classes - there is no macro-F1 "for Sitting". The per-class equivalents are
#  recall, precision and F1. Two one-vs-rest columns are included as well:
#     bal-acc  = (recall + specificity) / 2, treating the class as one-vs-rest
#     OvR acc  = (TP + TN) / total for that class against all others
#  OvR accuracy is reported because it is often asked for, but it is misleading
#  for rare classes: Running scores 0.96 there simply because 99.6% of segments
#  are not Running. Read F1 and recall instead.
# ---------------------------------------------------------------------------

def fold_predictions():
    """(y_true, y_pred) per fold - from memory if trained, else from disk."""
    if "results" in globals():
        return [(np.asarray(r["y_true"]), np.asarray(r["y_pred"])) for r in results]
    P = json.load(open(f"{RESULTS_DIR}/predictions.json"))
    return [(np.asarray(P[str(i)]["y_true"]), np.asarray(P[str(i)]["y_pred"]))
            for i in range(N_FOLDS)]


def confusion_np(y_true, y_pred, n=N_CLASSES):
    cm = np.zeros((n, n), dtype=np.int64)
    np.add.at(cm, (y_true, y_pred), 1)          # rows = true, cols = predicted
    return cm


folds = fold_predictions()
cm = sum(confusion_np(yt, yp) for yt, yp in folds)

tp = np.diag(cm).astype(np.float64)
support = cm.sum(1)                              # true instances per class
predicted = cm.sum(0)                            # predictions made per class
total = cm.sum()

recall = tp / np.maximum(support, 1)
precision = np.where(predicted > 0, tp / np.maximum(predicted, 1), 0.0)
f1 = np.where(precision + recall > 0,
              2 * precision * recall / np.maximum(precision + recall, 1e-12), 0.0)
specificity = (total - support - (predicted - tp)) / np.maximum(total - support, 1)
bal_acc = (recall + specificity) / 2             # one-vs-rest balanced accuracy
ovr_acc = (total - (support - tp) - (predicted - tp)) / total

print(f"TABLE 1 - per-class metrics, pooled over {N_FOLDS} folds "
      f"({total:,} test segments)\n")
print(f"{'idx':>3}  {'activity':22s} {'recall':>8s} {'precision':>10s} {'F1':>8s} "
      f"{'bal-acc':>9s} {'OvR acc':>9s} {'support':>11s}")
print("-" * 86)
for k in range(N_CLASSES):
    print(f"{k:>3}  {CLASS_NAMES[k]:22s} {recall[k]:8.3f} {precision[k]:10.3f} "
          f"{f1[k]:8.3f} {bal_acc[k]:9.3f} {ovr_acc[k]:9.3f} {support[k]:11,}")
print("-" * 86)
print(f"{'':>3}  {'macro average':22s} {recall.mean():8.3f} {precision.mean():10.3f} "
      f"{f1.mean():8.3f} {bal_acc.mean():9.3f}")
print(f"{'':>3}  {'overall accuracy':22s} {'':>8s} {'':>10s} {'':>8s} {'':>9s} "
      f"{tp.sum() / total:9.3f} {total:11,}")
print("\n  note: OvR accuracy flatters rare classes - Running reads 0.96 there "
      "while its F1 is 0.03,\n        because 99.6% of segments are not Running. "
      "F1 and recall are the honest columns.")

TABLE 1 - per-class metrics, pooled over 5 folds (1,333,415 test segments)

idx  activity                 recall  precision       F1   bal-acc   OvR acc     support
--------------------------------------------------------------------------------------
  0  Lying down                0.748      0.465    0.573     0.646     0.615     461,429
  1  Sitting                   0.172      0.575    0.265     0.536     0.579     588,134
  2  Walking                   0.444      0.413    0.428     0.698     0.916      94,632
  3  Running                   0.187      0.021    0.038     0.577     0.965       4,985
  4  Bicycling                 0.586      0.206    0.305     0.775     0.958      20,827
  5  Standing in place         0.204      0.054    0.085     0.553     0.882      35,709
  6  Standing and moving       0.126      0.215    0.159     0.539     0.872     127,699
--------------------------------------------------------------------------------------
     macro average             0.352  

In [10]:
# ---------------------------------------------------------------------------
#  TABLE 2 - per-class score for each fold separately
# ---------------------------------------------------------------------------
#  The spread across folds matters as much as the mean here. Only ~5 test users
#  per fold have Running or Bicycling, so which individuals land in which fold
#  moves these numbers more than the model does. A large std on a rare class is
#  a statement about the fold assignment, not about the classifier.
# ---------------------------------------------------------------------------

TABLE2_METRIC = "f1"          # "f1", "recall" or "precision"

per_fold = np.full((N_FOLDS, N_CLASSES), np.nan)
for i, (yt, yp) in enumerate(folds):
    c = confusion_np(yt, yp)
    d = np.diag(c).astype(np.float64)
    sup, pred = c.sum(1), c.sum(0)
    rec = np.where(sup > 0, d / np.maximum(sup, 1), np.nan)
    pre = np.where(pred > 0, d / np.maximum(pred, 1), 0.0)
    if TABLE2_METRIC == "recall":
        per_fold[i] = rec
    elif TABLE2_METRIC == "precision":
        per_fold[i] = np.where(sup > 0, pre, np.nan)
    else:
        r0 = np.nan_to_num(rec)
        per_fold[i] = np.where(sup > 0,
                               np.where(pre + r0 > 0,
                                        2 * pre * r0 / np.maximum(pre + r0, 1e-12), 0.0),
                               np.nan)

print(f"TABLE 2 - per-class {TABLE2_METRIC.upper()} across the {N_FOLDS} folds\n")
print(f"{'idx':>3}  {'activity':22s}"
      + "".join(f"{'fold ' + str(i):>9s}" for i in range(N_FOLDS))
      + f"{'mean':>9s}{'std':>8s}")
print("-" * 87)
for k in range(N_CLASSES):
    row = per_fold[:, k]
    print(f"{k:>3}  {CLASS_NAMES[k]:22s}"
          + "".join(f"{per_fold[i, k]:9.3f}" for i in range(N_FOLDS))
          + f"{np.nanmean(row):9.3f}{np.nanstd(row, ddof=1):8.3f}")
print("-" * 87)
print(f"{'':>3}  {'macro (mean of rows)':22s}"
      + "".join(f"{np.nanmean(per_fold[i]):9.3f}" for i in range(N_FOLDS))
      + f"{np.nanmean(per_fold):9.3f}")

worst = int(np.nanargmax(np.nanstd(per_fold, axis=0, ddof=1)))
lo, hi = np.nanmin(per_fold[:, worst]), np.nanmax(per_fold[:, worst])
print(f"\n  most fold-dependent class: {CLASS_NAMES[worst]} "
      f"({TABLE2_METRIC} ranges {lo:.3f} to {hi:.3f} across folds)")

TABLE 2 - per-class F1 across the 5 folds

idx  activity                 fold 0   fold 1   fold 2   fold 3   fold 4     mean     std
---------------------------------------------------------------------------------------
  0  Lying down                0.602    0.555    0.600    0.561    0.515    0.566   0.036
  1  Sitting                   0.236    0.206    0.360    0.259    0.161    0.245   0.074
  2  Walking                   0.495    0.349    0.446    0.356    0.582    0.446   0.098
  3  Running                   0.049    0.031    0.040    0.008    0.086    0.043   0.029
  4  Bicycling                 0.229    0.151    0.542    0.310    0.415    0.330   0.154
  5  Standing in place         0.126    0.092    0.065    0.064    0.098    0.089   0.026
  6  Standing and moving       0.073    0.197    0.207    0.118    0.152    0.149   0.056
---------------------------------------------------------------------------------------
     macro (mean of rows)      0.259    0.226    0.323    0.2